# 生徒AI設計 発表用実験Notebook

最終更新: 2026-07-26 JST

このNotebookは、生徒AIの設計を進捗報告で説明するための発表用Notebookです。

主役は認知モデルです。LLMは発話生成器として使いますが、正答・誤答や理解度の制御は外部の認知モデルで行います。

## このNotebookで見せるもの

1. 生徒AIの設計方針
2. 正答確率を決める認知モデルの式
3. 理解度とテスト結果の関係
4. 問題難易度、誤概念、スキル弱点の影響
5. 性格別の発話サンプル
6. 複数生徒AIにしたときに何が変わるか

補助実験は下部に置いています。発表では上から必要なセルだけ実行してください。


## 1. Setup

Colabで実行する場合は、先にGitHubからcloneしてからこのNotebookを開いてください。

~~~python
REPO_URL = "https://github.com/Hiromu-0219/student-ai-test.git"
!git clone {REPO_URL} /content/student-ai
%cd /content/student-ai
!pip install -q -r requirements.txt
~~~

ローカルやColab上で、現在の作業ディレクトリから `src` をimportできるようにします。


In [ ]:
from pathlib import Path
import os
import sys

REPO_URL = "https://github.com/Hiromu-0219/student-ai-test.git"
REPO_DIR = Path("/content/student-ai")

def find_project_root():
    candidates = [
        Path.cwd(),
        *Path.cwd().parents,
        REPO_DIR,
    ]
    for candidate in candidates:
        if (candidate / "src").exists() and (candidate / "data").exists():
            return candidate
    return None

PROJECT_ROOT = find_project_root()

if PROJECT_ROOT is None:
    if Path("/content").exists():
        print("Repository is not found. Clone it first by running:")
        print(f"!git clone {REPO_URL} {REPO_DIR}")
        raise RuntimeError("student-ai repository is not available. Run the clone command above, then rerun this cell.")
    raise RuntimeError("student-ai repository is not available. Please run this notebook from the repository root.")

os.chdir(PROJECT_ROOT)
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

print("project_root:", PROJECT_ROOT)
print("current_dir:", Path.cwd())
print("src import path ready:", (PROJECT_ROOT / "src").exists())


## 2. 発表での主張

本研究で作っている生徒AIは、人間生徒の完全な代替ではありません。

主張する範囲は次です。

> 一次方程式の学習場面において、理解度・誤概念・個人特徴を外部状態として制御した生徒AIが、教育シミュレーション内の限定的な学習者代理として使えるかを、内部妥当性から検証する。

設計は次の3層です。

| 層 | 役割 |
| --- | --- |
| 認知モデル | 理解度、問題難易度、誤概念、guess/slipから正答確率を決める |
| 個人特徴モデル | 自己効力感、質問傾向、意欲、Big Fiveから発話スタイルを決める |
| LLM発話生成器 | 認知モデルの結果と個人特徴をもとに、生徒らしい返答を生成する |


## 3. 認知モデルの式

生徒 \(i\)、問題 \(j\)、対象スキル \(k(j)\) とします。

まず、スキル別理解度と単元全体の理解度から、問題に対する能力値を作ります。

$$
\theta_{i,j} = 0.85 s_{i,k(j)} + 0.15 g_i
$$

次に、BKT的な考え方として、未習得でも偶然正答する \(guess\)、習得済みでもミスする \(slip\) を入れます。

$$
P_{BKT}(correct_{i,j}) =
m_{i,j}(1 - slip_{i,j}) + (1 - m_{i,j})guess_j
$$

さらに、IRT的な考え方として、生徒の能力値 \(\theta\) と問題難易度 \(d\) の差を使います。

$$
P_{IRT}(correct_{i,j}) =
guess_j + (1 - guess_j - slip_{i,j})
\sigma\left(\frac{\theta_{i,j} - d_j}{12}\right)
$$

最終的な正答確率は、BKT的成分とIRT的成分を混ぜ、自己効力感・意欲・誤概念ペナルティを加えて決めます。

$$
P(correct_{i,j}) =
clip\left(
0.45P_{BKT} + 0.55P_{IRT}
+ A_i - M_{i,j},\ 0.05,\ 0.95
\right)
$$

| 記号 | 意味 |
| --- | --- |
| \(s_{i,k}\) | 生徒iのスキルkの理解度 |
| \(g_i\) | 生徒iの一次方程式全体理解度 |
| \(d_j\) | 問題jの難易度 |
| \(guess_j\) | 低理解でも偶然正答する確率 |
| \(slip_{i,j}\) | 理解していてもミスする確率 |
| \(A_i\) | 自己効力感・意欲による補正 |
| \(M_{i,j}\) | 誤概念ペナルティ |


## 4. 発表用実験を実行

このセルではLLMを呼びません。認知モデルとmock発話生成で、発表に必要な結果をまとめて作ります。

出力されるもの:

- 学習曲線
- 難易度別テスト結果
- 誤概念あり/なしの差
- スキル弱点の影響
- 性格別発話サンプル
- 共有用txt


In [ ]:
from pathlib import Path
import os
import sys

if "PROJECT_ROOT" not in globals() or not (Path(PROJECT_ROOT) / "src").exists():
    candidates = [Path.cwd(), *Path.cwd().parents, Path("/content/student-ai")]
    PROJECT_ROOT = next(
        (
            candidate
            for candidate in candidates
            if (candidate / "src").exists() and (candidate / "data").exists()
        ),
        None,
    )
    if PROJECT_ROOT is None:
        raise RuntimeError("src が見つかりません。Setupセルを実行するか、/content/student-ai にcloneしてください。")
    os.chdir(PROJECT_ROOT)
    if str(PROJECT_ROOT) not in sys.path:
        sys.path.insert(0, str(PROJECT_ROOT))

from pprint import pprint
from pathlib import Path

import pandas as pd
import matplotlib.pyplot as plt
from IPython.display import display, Markdown

from src.experiment import (
    export_student_ai_evaluation,
    export_student_ai_evaluation_for_codex,
    run_student_ai_evaluation,
)

STUDENT_ID = "S002"
TEST_ID = "linear_equation_20q_001"
COGNITIVE_MODEL_TYPE = "bkt_irt"
UNDERSTANDING_LEVELS = list(range(0, 101, 10))
USE_MOCK_MODEL = True

OUTPUT_DIR = PROJECT_ROOT / "data" / "assessments" / "presentation"
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

student_ai_result = run_student_ai_evaluation(
    student_id=STUDENT_ID,
    test_id=TEST_ID,
    understanding_levels=UNDERSTANDING_LEVELS,
    use_mock_model=USE_MOCK_MODEL,
    cognitive_model_type=COGNITIVE_MODEL_TYPE,
)

summary_path = export_student_ai_evaluation(student_ai_result)
codex_path = export_student_ai_evaluation_for_codex(student_ai_result)

print("summary_path:", summary_path)
print("codex_share_path:", codex_path)
print("cognitive_model:", student_ai_result["cognitive_model"])
print("question_count:", student_ai_result["question_count"])
print()
pprint(student_ai_result["summary"])


## 5. テスト結果: 理解度と正答率

ここはスライドに貼る中心グラフです。

見ること:

- 理解度が上がると、正答率と平均正答確率が上がる
- ただし、低理解でもguessにより0%にはならない
- 高理解でもslipにより100%にはならない


In [ ]:
learning_curve_df = pd.DataFrame(student_ai_result["learning_curve"])
display(learning_curve_df)

fig, ax1 = plt.subplots(figsize=(8, 5))

ax1.plot(
    learning_curve_df["understanding"],
    learning_curve_df["accuracy"] * 100,
    marker="o",
    linewidth=2,
    label="observed accuracy",
)
ax1.plot(
    learning_curve_df["understanding"],
    learning_curve_df["average_correct_probability"],
    marker="s",
    linewidth=2,
    label="average correct probability",
)
ax1.set_title("Understanding vs Test Performance")
ax1.set_xlabel("understanding score")
ax1.set_ylabel("percent")
ax1.set_xlim(0, 100)
ax1.set_ylim(0, 100)
ax1.grid(True, alpha=0.3)
ax1.legend()

learning_curve_path = OUTPUT_DIR / "student_ai_learning_curve.png"
plt.tight_layout()
plt.savefig(learning_curve_path, dpi=200, bbox_inches="tight")
plt.show()

print("saved:", learning_curve_path)


## 6. テスト結果: 難易度・誤概念・スキル弱点

この図は、単に「理解度=正答率」としているわけではないことを見せるための図です。

見ること:

- 同じ理解度でも、難しい問題ほど正答確率が下がる
- 誤概念があると、関連問題で正答確率が下がる
- 弱点スキルを変えると、対応する問題の正答確率が下がる


In [ ]:
difficulty_df = pd.DataFrame(student_ai_result["difficulty_breakdown"])
misconception_df = pd.DataFrame(student_ai_result["misconception_comparison"]["rows"])
skill_df = pd.DataFrame(student_ai_result["skill_breakdown"])

display(Markdown("### Difficulty Breakdown"))
display(difficulty_df)
display(Markdown("### Misconception Comparison"))
display(misconception_df)
display(Markdown("### Skill Weakness Breakdown"))
display(skill_df)

fig, axes = plt.subplots(1, 3, figsize=(18, 5))

axes[0].bar(
    difficulty_df["label"],
    difficulty_df["average_correct_probability"],
    color="#4C78A8",
)
axes[0].set_title("Difficulty Effect")
axes[0].set_xlabel("difficulty")
axes[0].set_ylabel("avg correct probability")
axes[0].set_ylim(0, 100)
axes[0].grid(True, axis="y", alpha=0.3)

axes[1].plot(
    misconception_df["understanding"],
    misconception_df["related_probability_gap"],
    marker="o",
    linewidth=2,
    color="#F58518",
)
axes[1].set_title("Misconception Effect")
axes[1].set_xlabel("understanding")
axes[1].set_ylabel("probability gap on related items")
axes[1].set_ylim(0, max(15, misconception_df["related_probability_gap"].max() + 3))
axes[1].grid(True, alpha=0.3)

axes[2].barh(
    skill_df["weak_skill"],
    skill_df["target_probability_drop"],
    color="#54A24B",
)
axes[2].set_title("Skill-specific Weakness")
axes[2].set_xlabel("probability drop")
axes[2].grid(True, axis="x", alpha=0.3)

plt.tight_layout()
effect_path = OUTPUT_DIR / "student_ai_cognitive_effects.png"
plt.savefig(effect_path, dpi=200, bbox_inches="tight")
plt.show()

print("saved:", effect_path)


## 7. 性格別の発話サンプル

同じ問題・同じ正答条件でも、個人特徴によって発話が変わることを見せます。

認知モデルが正答方針を決め、個人特徴モデルが発話スタイルを変えます。


In [ ]:
utterance_rows = []
for sample in student_ai_result["utterance_samples"]:
    features = sample["utterance_features"]
    profile = sample["personality_profile"]
    utterance_rows.append({
        "profile_id": sample["profile_id"],
        "utterance": sample["utterance"],
        "confidence_expression": profile.get("confidence_expression"),
        "question_behavior": profile.get("question_behavior"),
        "motivation_expression": profile.get("motivation_expression"),
        "char_count": features.get("char_count"),
        "question_mark_count": features.get("question_mark_count"),
        "uncertainty_marker_count": features.get("uncertainty_marker_count"),
    })

utterance_df = pd.DataFrame(utterance_rows)
display(utterance_df)

feature_df = utterance_df.set_index("profile_id")[
    ["char_count", "question_mark_count", "uncertainty_marker_count"]
]

ax = feature_df.plot(kind="bar", figsize=(9, 5))
ax.set_title("Observable Utterance Features by Personality Profile")
ax.set_xlabel("personality profile")
ax.set_ylabel("count")
ax.grid(True, axis="y", alpha=0.3)
plt.xticks(rotation=20, ha="right")
plt.tight_layout()

utterance_feature_path = OUTPUT_DIR / "student_ai_personality_utterance_features.png"
plt.savefig(utterance_feature_path, dpi=200, bbox_inches="tight")
plt.show()

print("saved:", utterance_feature_path)


## 8. 複数生徒AIにすると何が変わるか

複数生徒AIでは、正答確率の式そのものは変わりません。

変わるのは、生徒ごとのパラメータ分布です。

~~~text
1人の生徒AI:
  P(correct) = f(その生徒の理解度, 誤概念, 個人特徴, 問題難易度)

複数生徒AI:
  class profile = { P(correct_1), P(correct_2), ..., P(correct_N) } の分布
~~~

発表では、「クラス全体の授業設計に使える情報が増える」と説明します。


In [ ]:
from src.class_manager import ClassManager
from src.cognitive_model import create_cognitive_model
from src.test_bank import TestBank

CLASS_ID = "class_20_mixed"

class_manager = ClassManager()
class_summary = class_manager.summarize_class(CLASS_ID)
student_states = class_manager.load_students(CLASS_ID)
test_data = TestBank().load_test(TEST_ID)
cognitive_model = create_cognitive_model(COGNITIVE_MODEL_TYPE)

skill_columns = [
    "score",
    "can_solve_ax_plus_b_equals_c",
    "can_transpose_terms",
    "can_divide_by_coefficient",
    "can_handle_negative_numbers",
    "can_handle_fractions",
]

student_rows = []
for state in student_states:
    linear_state = state["knowledge_state"]["linear_equation"]
    row = {
        "student_id": state["student_id"],
        "self_efficacy": state["self_efficacy"],
        "question_tendency": state["question_tendency"],
        "motivation": state["motivation"],
        "misconception_count": len(state.get("misconceptions", [])),
    }
    for skill in skill_columns:
        row[skill] = linear_state.get(skill, 0)
    student_rows.append(row)

student_df = pd.DataFrame(student_rows).sort_values("score").reset_index(drop=True)

probability_rows = []
for state in student_states:
    for question in test_data["questions"]:
        directive = cognitive_model.build_assessment_directive(
            student_state=state,
            question=question,
        )
        probability_rows.append({
            "student_id": state["student_id"],
            "difficulty": question["difficulty"],
            "skill": question["skill"],
            "correct_probability": directive["correct_probability"],
        })

probability_df = pd.DataFrame(probability_rows)
student_probability_df = (
    probability_df.groupby("student_id", as_index=False)["correct_probability"].mean()
    .rename(columns={"correct_probability": "average_correct_probability"})
    .merge(student_df[["student_id", "score"]], on="student_id")
    .sort_values("score")
)

display(Markdown("### Class Summary"))
display(pd.DataFrame([class_summary]))
display(Markdown("### Student Parameter Table"))
display(student_df)

fig, axes = plt.subplots(1, 3, figsize=(18, 5))

axes[0].hist(student_df["score"], bins=[0, 20, 40, 60, 80, 100], edgecolor="black", color="#4C78A8")
axes[0].set_title("Class Understanding Distribution")
axes[0].set_xlabel("overall understanding score")
axes[0].set_ylabel("student count")
axes[0].set_xlim(0, 100)
axes[0].grid(True, alpha=0.3)

axes[1].bar(student_probability_df["student_id"], student_probability_df["average_correct_probability"], color="#F58518")
axes[1].set_title("Predicted Accuracy by Student")
axes[1].set_xlabel("student")
axes[1].set_ylabel("average correct probability")
axes[1].set_ylim(0, 100)
axes[1].tick_params(axis="x", rotation=90)
axes[1].grid(True, axis="y", alpha=0.3)

skill_matrix = student_df.set_index("student_id")[skill_columns]
im = axes[2].imshow(skill_matrix, aspect="auto", vmin=0, vmax=100, cmap="viridis")
axes[2].set_title("Skill Profile Heatmap")
axes[2].set_xlabel("skill")
axes[2].set_ylabel("student")
axes[2].set_xticks(range(len(skill_columns)))
axes[2].set_xticklabels(skill_columns, rotation=45, ha="right")
axes[2].set_yticks(range(len(skill_matrix.index)))
axes[2].set_yticklabels(skill_matrix.index)
fig.colorbar(im, ax=axes[2], label="understanding score")

plt.tight_layout()
class_path = OUTPUT_DIR / "student_ai_class_distribution.png"
plt.savefig(class_path, dpi=200, bbox_inches="tight")
plt.show()

print("saved:", class_path)


## 9. 発表で使うまとめ

このセルは、口頭説明用の短いまとめを表示します。


In [ ]:
display(Markdown(f"""
## Presentation Summary

- 生徒AIは、LLMに正誤を任せるのではなく、外部の認知モデルで正答確率を制御する。
- BKT/IRT寄りモデルにより、理解度、問題難易度、guess、slip、誤概念ペナルティを分けて説明できる。
- テスト結果では、理解度の上昇に伴い正答率・平均正答確率が上昇する。
- 性格別発話では、同じ認知状態でも、自信、質問傾向、意欲によって観察可能な発話特徴が変わる。
- 複数生徒AIでは、式が変わるのではなく、生徒ごとのパラメータ分布が生まれ、クラス全体の授業設計に使える。

共有用txt:

- {codex_path}

保存した図:

- {learning_curve_path}
- {effect_path}
- {utterance_feature_path}
- {class_path}
"""))


--- 

# 補助実験

ここから下は、発表で時間があれば使う補助実験です。


## A1. 従来モデルとBKT/IRT寄りモデルの比較

従来モデルとBKT/IRT寄りモデルを比較します。

発表では、「なぜBKT/IRT寄りにしたのか」を説明するときだけ使います。


In [ ]:
from src.experiment import (
    compare_cognitive_models,
    export_cognitive_model_comparison_for_codex,
)

cognitive_model_comparison = compare_cognitive_models(
    student_id=STUDENT_ID,
    test_id=TEST_ID,
    understanding_levels=UNDERSTANDING_LEVELS,
    use_mock_model=True,
)

comparison_path = export_cognitive_model_comparison_for_codex(cognitive_model_comparison)
comparison_df = pd.DataFrame(cognitive_model_comparison["learning_curve_comparison"])

print("comparison_path:", comparison_path)
display(comparison_df)

plt.figure(figsize=(8, 5))
plt.plot(
    comparison_df["understanding"],
    comparison_df["legacy_average_correct_probability"],
    marker="o",
    label="legacy",
)
plt.plot(
    comparison_df["understanding"],
    comparison_df["bkt_irt_average_correct_probability"],
    marker="s",
    label="bkt_irt",
)
plt.title("Legacy vs BKT/IRT-inspired Cognitive Model")
plt.xlabel("understanding")
plt.ylabel("average correct probability")
plt.ylim(0, 100)
plt.grid(True, alpha=0.3)
plt.legend()
plt.tight_layout()

comparison_figure_path = OUTPUT_DIR / "student_ai_cognitive_model_comparison.png"
plt.savefig(comparison_figure_path, dpi=200, bbox_inches="tight")
plt.show()

print("saved:", comparison_figure_path)


## A2. LLM発話確認は必要なときだけ

発表準備では、通常このセルは実行しません。

理由:

- ColabでLLMロードに時間がかかる
- 認知モデルの検証にはLLMロードは不要
- 発話自然性を確認したいときだけ別途 `use_mock_model=False` で実行する
